In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()

yellow_preview = con.execute("""
    SELECT *
    FROM read_parquet('../data/bronze/yellow_tripdata_2025-01.parquet')
    LIMIT 5
""").df()

green_preview = con.execute("""
    SELECT *
    FROM read_parquet('../data/bronze/green_tripdata_2025-01.parquet')
    LIMIT 5
""").df()

yellow_preview

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
yellow_cols = con.execute("""
    DESCRIBE SELECT *
    FROM read_parquet('../data/bronze/yellow_tripdata_2025-01.parquet')
""").df()

green_cols = con.execute("""
    DESCRIBE SELECT *
    FROM read_parquet('../data/bronze/green_tripdata_2025-01.parquet')
""").df()

print("YELLOW COLUMNS")
display(yellow_cols)

print("GREEN COLUMNS")
display(green_cols)

YELLOW COLUMNS


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


GREEN COLUMNS


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [5]:
zone_lookup = con.execute("""
    SELECT *
    FROM read_csv_auto('../data/static/taxi_zone_lookup.csv')
    LIMIT 10
""").df()

zone_lookup

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [6]:
query = """
WITH yellow AS (
    SELECT
        'Yellow' AS service_type,
        PULocationID AS pickup_location_id,
        COUNT(*) AS trip_count,
        SUM(total_amount) AS total_revenue,
        AVG(fare_amount) AS avg_fare,
        AVG(tip_amount) AS avg_tip
    FROM read_parquet('../data/bronze/yellow_tripdata_2025-*.parquet')
    GROUP BY PULocationID
),
green AS (
    SELECT
        'Green' AS service_type,
        PULocationID AS pickup_location_id,
        COUNT(*) AS trip_count,
        SUM(total_amount) AS total_revenue,
        AVG(fare_amount) AS avg_fare,
        AVG(tip_amount) AS avg_tip
    FROM read_parquet('../data/bronze/green_tripdata_2025-*.parquet')
    GROUP BY PULocationID
),
combined AS (
    SELECT * FROM yellow
    UNION ALL
    SELECT * FROM green
)
SELECT
    c.service_type,
    c.pickup_location_id,
    z.Zone AS zone,
    z.Borough AS borough,
    c.trip_count,
    ROUND(c.total_revenue, 2) AS total_revenue,
    ROUND(c.avg_fare, 2) AS avg_fare,
    ROUND(c.avg_tip, 2) AS avg_tip
FROM combined c
LEFT JOIN read_csv_auto('../data/static/taxi_zone_lookup.csv') z
    ON c.pickup_location_id = z.LocationID
ORDER BY c.service_type, total_revenue DESC
"""

df_zone_profit = con.execute(query).df()
df_zone_profit.head(20)

,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Green,74,East Harlem North,Manhattan,47165,984335.32,14.82,2.69
1,Green,75,East Harlem South,Manhattan,31409,666446.63,14.44,2.52
2,Green,43,Central Park,Manhattan,10292,235661.07,14.94,3.05
3,Green,166,Morningside Heights,Manhattan,9967,220880.63,15.61,2.86
4,Green,95,Forest Hills,Queens,9094,190909.87,16.28,1.88
5,Green,82,Elmhurst,Queens,7282,173034.26,17.99,1.95
6,Green,97,Fort Greene,Brooklyn,7251,165564.11,17.35,2.84
7,Green,244,Washington Heights South,Manhattan,4757,163616.04,26.17,4.02
8,Green,130,Jamaica,Queens,5804,161709.92,22.37,2.63
9,Green,41,Central Harlem,Manhattan,8122,156814.86,14.35,1.86


In [8]:
import os

os.makedirs('../data/gold', exist_ok=True)

df_zone_profit.to_csv('../data/gold/zone_profitability.csv', index=False)
print("Saved to ../data/gold/zone_profitability.csv")

Saved to ../data/gold/zone_profitability.csv


In [9]:
df_zone_profit[df_zone_profit["service_type"] == "Yellow"].head(10)

,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
242,Yellow,132,JFK Airport,Queens,593786,43031383.05,55.64,8.74
243,Yellow,138,LaGuardia Airport,Queens,392095,26260003.96,43.12,8.98
244,Yellow,161,Midtown Center,Manhattan,697146,16662077.65,15.09,3.00
245,Yellow,237,Upper East Side South,Manhattan,677462,13443828.48,12.21,2.55
246,Yellow,230,Times Sq/Theatre District,Manhattan,511371,13215385.81,16.73,3.15
247,Yellow,236,Upper East Side North,Manhattan,619288,12508276.37,12.74,2.54
248,Yellow,186,Penn Station/Madison Sq West,Manhattan,493230,11518128.75,15.03,2.97
249,Yellow,162,Midtown East,Manhattan,485199,11303663.78,14.56,3.01
250,Yellow,163,Midtown North,Manhattan,402697,9572507.51,15.04,3.02
251,Yellow,142,Lincoln Square East,Manhattan,448797,9507621.33,13.34,2.67


In [10]:
df_zone_profit[df_zone_profit["service_type"] == "Green"].head(10)

,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Green,74,East Harlem North,Manhattan,47165,984335.32,14.82,2.69
1,Green,75,East Harlem South,Manhattan,31409,666446.63,14.44,2.52
2,Green,43,Central Park,Manhattan,10292,235661.07,14.94,3.05
3,Green,166,Morningside Heights,Manhattan,9967,220880.63,15.61,2.86
4,Green,95,Forest Hills,Queens,9094,190909.87,16.28,1.88
5,Green,82,Elmhurst,Queens,7282,173034.26,17.99,1.95
6,Green,97,Fort Greene,Brooklyn,7251,165564.11,17.35,2.84
7,Green,244,Washington Heights South,Manhattan,4757,163616.04,26.17,4.02
8,Green,130,Jamaica,Queens,5804,161709.92,22.37,2.63
9,Green,41,Central Harlem,Manhattan,8122,156814.86,14.35,1.86


In [11]:
print("TOP 10 YELLOW BY TOTAL REVENUE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Yellow"]
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

print("TOP 10 GREEN BY TOTAL REVENUE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Green"]
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

TOP 10 YELLOW BY TOTAL REVENUE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
242,Yellow,132,JFK Airport,Queens,593786,43031383.05,55.64,8.74
243,Yellow,138,LaGuardia Airport,Queens,392095,26260003.96,43.12,8.98
244,Yellow,161,Midtown Center,Manhattan,697146,16662077.65,15.09,3.00
245,Yellow,237,Upper East Side South,Manhattan,677462,13443828.48,12.21,2.55
246,Yellow,230,Times Sq/Theatre District,Manhattan,511371,13215385.81,16.73,3.15
247,Yellow,236,Upper East Side North,Manhattan,619288,12508276.37,12.74,2.54
248,Yellow,186,Penn Station/Madison Sq West,Manhattan,493230,11518128.75,15.03,2.97
249,Yellow,162,Midtown East,Manhattan,485199,11303663.78,14.56,3.01
250,Yellow,163,Midtown North,Manhattan,402697,9572507.51,15.04,3.02
251,Yellow,142,Lincoln Square East,Manhattan,448797,9507621.33,13.34,2.67


TOP 10 GREEN BY TOTAL REVENUE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Green,74,East Harlem North,Manhattan,47165,984335.32,14.82,2.69
1,Green,75,East Harlem South,Manhattan,31409,666446.63,14.44,2.52
2,Green,43,Central Park,Manhattan,10292,235661.07,14.94,3.05
3,Green,166,Morningside Heights,Manhattan,9967,220880.63,15.61,2.86
4,Green,95,Forest Hills,Queens,9094,190909.87,16.28,1.88
5,Green,82,Elmhurst,Queens,7282,173034.26,17.99,1.95
6,Green,97,Fort Greene,Brooklyn,7251,165564.11,17.35,2.84
7,Green,244,Washington Heights South,Manhattan,4757,163616.04,26.17,4.02
8,Green,130,Jamaica,Queens,5804,161709.92,22.37,2.63
9,Green,41,Central Harlem,Manhattan,8122,156814.86,14.35,1.86


In [12]:
print("TOP 10 YELLOW BY AVG FARE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Yellow"]
    .sort_values("avg_fare", ascending=False)
    .head(10)
)

print("TOP 10 GREEN BY AVG FARE")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Green"]
    .sort_values("avg_fare", ascending=False)
    .head(10)
)

TOP 10 YELLOW BY AVG FARE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
493,Yellow,204,Rossville/Woodrow,Staten Island,9,1183.18,110.18,11.19
359,Yellow,1,Newark Airport,EWR,1529,146114.41,82.80,9.70
307,Yellow,265,Outside of NYC,N/A,5728,526875.50,80.01,8.56
502,Yellow,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,1,90.69,70.00,9.00
242,Yellow,132,JFK Airport,Queens,593786,43031383.05,55.64,8.74
497,Yellow,84,Eltingville/Annadale/Prince's Bay,Staten Island,12,744.01,53.63,1.61
495,Yellow,44,Charleston/Tottenville,Staten Island,13,922.47,51.82,4.45
494,Yellow,109,Great Kills,Staten Island,18,1092.45,48.63,3.01
353,Yellow,93,Flushing Meadows-Corona Park,Queens,2734,163296.13,48.01,6.14
487,Yellow,206,Saint George/New Brighton,Staten Island,44,2619.50,47.33,1.24


TOP 10 GREEN BY AVG FARE


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
181,Green,1,Newark Airport,EWR,5,567.50,94.40,18.00
215,Green,245,West Brighton,Staten Island,2,180.98,67.30,12.50
233,Green,6,Arrochar/Fort Wadsworth,Staten Island,1,73.33,61.39,0.00
45,Green,265,Outside of NYC,N/A,121,8376.60,61.38,5.90
156,Green,86,Far Rockaway,Queens,16,984.84,58.78,0.13
148,Green,117,Hammels/Arverne,Queens,20,1145.71,52.99,0.35
183,Green,154,Marine Park/Floyd Bennett Field,Brooklyn,10,541.92,51.94,0.00
64,Green,219,Springfield Gardens South,Queens,71,4491.01,50.00,4.86
237,Green,176,Oakwood,Staten Island,1,63.18,47.80,0.00
29,Green,93,Flushing Meadows-Corona Park,Queens,480,27197.36,47.32,5.39


In [13]:
print("TOP 10 YELLOW BY AVG TIP")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Yellow"]
    .sort_values("avg_tip", ascending=False)
    .head(10)
)

print("TOP 10 GREEN BY AVG TIP")
display(
    df_zone_profit[df_zone_profit["service_type"] == "Green"]
    .sort_values("avg_tip", ascending=False)
    .head(10)
)

TOP 10 YELLOW BY AVG TIP


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
493,Yellow,204,Rossville/Woodrow,Staten Island,9,1183.18,110.18,11.19
359,Yellow,1,Newark Airport,EWR,1529,146114.41,82.80,9.70
502,Yellow,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,1,90.69,70.00,9.00
243,Yellow,138,LaGuardia Airport,Queens,392095,26260003.96,43.12,8.98
242,Yellow,132,JFK Airport,Queens,593786,43031383.05,55.64,8.74
307,Yellow,265,Outside of NYC,N/A,5728,526875.50,80.01,8.56
282,Yellow,70,East Elmhurst,Queens,43605,2613294.02,38.84,8.03
500,Yellow,199,Rikers Island,Bronx,8,418.34,34.59,7.37
492,Yellow,2,Jamaica Bay,Queens,23,1274.54,42.94,7.15
353,Yellow,93,Flushing Meadows-Corona Park,Queens,2734,163296.13,48.01,6.14


TOP 10 GREEN BY AVG TIP


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
181,Green,1,Newark Airport,EWR,5,567.50,94.40,18.00
215,Green,245,West Brighton,Staten Island,2,180.98,67.30,12.50
239,Green,224,Stuy Town/Peter Cooper Village,Manhattan,1,38.80,27.33,6.47
45,Green,265,Outside of NYC,N/A,121,8376.60,61.38,5.90
73,Green,138,LaGuardia Airport,Queens,74,3580.03,36.84,5.87
43,Green,146,Long Island City/Queens Plaza,Queens,300,9716.36,22.51,5.76
29,Green,93,Flushing Meadows-Corona Park,Queens,480,27197.36,47.32,5.39
63,Green,34,Brooklyn Navy Yard,Brooklyn,91,4636.51,43.29,5.03
25,Green,157,Maspeth,Queens,625,28989.08,38.14,4.96
169,Green,194,Randalls Island,Manhattan,17,751.21,30.71,4.90


In [1]:
import json
import pandas as pd

with open("../data/static/taxi_zones.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)

print("Jumlah feature:", len(geo["features"]))
print("\nContoh properties feature pertama:")
print(geo["features"][0]["properties"])

Jumlah feature: 263

Contoh properties feature pertama:
{'shape_area': '0.0007823067885', 'objectid': '1', 'shape_leng': '0.116357453189', 'location_id': '1', 'zone': 'Newark Airport', 'borough': 'EWR'}


In [2]:
lookup = pd.read_csv("../data/static/taxi_zone_lookup.csv")

print("Kolom lookup:")
print(lookup.columns.tolist())

print("\n5 baris pertama lookup:")
display(lookup.head())

Kolom lookup:
['LocationID', 'Borough', 'Zone', 'service_zone']

5 baris pertama lookup:


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [3]:
zone_profit = pd.read_csv("../data/gold/zone_profitability.csv")

print("Kolom zone_profitability:")
print(zone_profit.columns.tolist())

display(zone_profit.head())

Kolom zone_profitability:
['service_type', 'pickup_location_id', 'zone', 'borough', 'trip_count', 'total_revenue', 'avg_fare', 'avg_tip']


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Green,74,East Harlem North,Manhattan,47165,984335.32,14.82,2.69
1,Green,75,East Harlem South,Manhattan,31409,666446.63,14.44,2.52
2,Green,43,Central Park,Manhattan,10292,235661.07,14.94,3.05
3,Green,166,Morningside Heights,Manhattan,9967,220880.63,15.61,2.86
4,Green,95,Forest Hills,Queens,9094,190909.87,16.28,1.88


In [4]:
import json
import copy

lookup = pd.read_csv("../data/static/taxi_zone_lookup.csv")
zone_profit = pd.read_csv("../data/gold/zone_profitability.csv")

# Kalau belum ada pickup_location_id, ambil dari lookup pakai zone + borough
if "pickup_location_id" not in zone_profit.columns:
    zone_profit = zone_profit.merge(
        lookup[["LocationID", "Borough", "Zone"]],
        left_on=["borough", "zone"],
        right_on=["Borough", "Zone"],
        how="left"
    )
    zone_profit = zone_profit.rename(columns={"LocationID": "pickup_location_id"})

print("Kolom akhir zone_profit:")
print(zone_profit.columns.tolist())

print("\nJumlah baris tanpa pickup_location_id:")
print(zone_profit["pickup_location_id"].isna().sum())

display(zone_profit.head())

Kolom akhir zone_profit:
['service_type', 'pickup_location_id', 'zone', 'borough', 'trip_count', 'total_revenue', 'avg_fare', 'avg_tip']

Jumlah baris tanpa pickup_location_id:
0


,service_type,pickup_location_id,zone,borough,trip_count,total_revenue,avg_fare,avg_tip
0,Green,74,East Harlem North,Manhattan,47165,984335.32,14.82,2.69
1,Green,75,East Harlem South,Manhattan,31409,666446.63,14.44,2.52
2,Green,43,Central Park,Manhattan,10292,235661.07,14.94,3.05
3,Green,166,Morningside Heights,Manhattan,9967,220880.63,15.61,2.86
4,Green,95,Forest Hills,Queens,9094,190909.87,16.28,1.88


In [5]:
zone_profit.to_csv("../data/gold/zone_profitability.csv", index=False)
print("zone_profitability.csv berhasil di-update")

zone_profitability.csv berhasil di-update
